# Hands-on 39 (Bab 15): Menambahkan Notifikasi Sederhana

Buku Referensi: *IoT Terintegrasi dengan Machine Learning & Data Analytics* (Onno W. Purbo), Bab 15.

Sistem tidak hanya mencatat anomali, tetapi mengabarkannya. Dua kanal sederhana sesuai
buku: publish ke topik `capstone/iotnode01/alert` (sudah dibangun di Hands-on 38) dan
listener notifikasi yang menampilkan alert serta menyimpannya ke CSV - titik ekstensi
menuju Telegram/email/WA.

In [1]:
import os
from pathlib import Path

while not (Path.cwd() / "scripts" / "mqtt").is_dir():
    os.chdir("..")
print("[INFO] Working directory:", Path.cwd())

[INFO] Working directory: /home/sandbox/iot-ml-book


## 1. Jalankan Sistem dengan Notifikasi

In [2]:
import subprocess, sys, time

broker = subprocess.Popen([sys.executable, "scripts/mqtt/run_broker.py"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)
try:
    listener = subprocess.Popen(
        [sys.executable, "scripts/capstone_alert_listener.py",
         "--max-messages", "2", "--timeout", "45"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    sub = subprocess.Popen(
        [sys.executable, "scripts/subscribers/capstone_subscriber_with_alert.py",
         "--max-messages", "30", "--timeout", "45"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(1)
    subprocess.run([sys.executable, "scripts/publishers/capstone_publisher.py",
                    "--count", "30", "--interval", "0.3", "--seed", "7"],
                   capture_output=True, text=True)
    out_listener, _ = listener.communicate()
    print("--- Output listener notifikasi ---")
    print(out_listener)
    sub.wait()
finally:
    broker.terminate()

--- Output listener notifikasi ---
[INFO] Menunggu alert di topik: capstone/iotnode01/alert

=== NOTIFIKASI ALERT ===
Waktu   : 2026-10-22T10:54:33
Device  : iotnode01
Alasan  : isolation_forest
Daya    : 234.52 W
Arus    : 5.24 A

=== NOTIFIKASI ALERT ===
Waktu   : 2026-10-22T10:54:35
Device  : iotnode01
Alasan  : threshold+isolation_forest
Daya    : 254.18158565343953 W
Arus    : 10.72371259863844 A

[SUKSES] 2 notifikasi disimpan ke outputs/reports/capstone_notifications.csv



## 2. Verifikasi Log Notifikasi

In [3]:
import pandas as pd
notif = pd.read_csv("outputs/reports/capstone_notifications.csv")
print(notif.to_string(index=False))

          timestamp device_id               alert_reason  temperature_c  humidity_pct  power_watt  current_amp
2026-10-22T10:54:33 iotnode01           isolation_forest          33.76         41.63  234.520000     5.240000
2026-10-22T10:54:35 iotnode01 threshold+isolation_forest          28.18         66.50  254.181586    10.723713


## Penutup

Notifikasi sampai ke listener melalui topik alert dan tersimpan di
`outputs/reports/capstone_notifications.csv`. Listener ini adalah titik ekstensi yang
jelas: ganti aksi cetak/simpan dengan pengiriman ke Telegram, email, atau WhatsApp.
Di Grafana, alert bisa ditampilkan dari measurement `capstone_alerts` saat memakai
storage InfluxDB (`--influx`).